# Network

> What nodes send and receive through: it ties the topology, protocol, backend and metrics together.

In [ ]:
#| default_exp network

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

from __future__ import annotations

import heapq
import itertools
from collections import defaultdict
from dataclasses import replace
from typing import Dict, List

from comm_core.backend import CommunicationBackend
from comm_core.core import Message, Node, TransmissionResult
from comm_core.metrics import MetricsCollector
from comm_core.protocol import Protocol
from comm_core.topology import Topology

In [ ]:
#| export

class Network:
    "What nodes send and receive through: the topology, protocol, backend and metrics together."

    def __init__(
        self,
        nodes: List[Node], # The nodes
        topology: Topology, # Who can communicate with whom
        protocol: Protocol, # When messages are transmitted
        backend: CommunicationBackend, # How a step's transmissions are simulated
        metrics: MetricsCollector | None = None, # What communication costs; None: a new collector
        delayed_delivery: bool = False, # Deliver a message at its transmission's start + latency (False: in the step it is sent)
    ):
        self.nodes = {
            node.id: node
            for node in nodes
        }

        self.topology = topology
        self.protocol = protocol
        self.backend = backend

        self.metrics = (
            metrics
            if metrics is not None
            else MetricsCollector()
        )

        self.delayed_delivery = delayed_delivery

        self._inbox: Dict[str, List[Message]] = (
            defaultdict(list)
        )

        self._timed: List[tuple] = []          # (time, order, message): messages sent for later
        self._in_flight: List[tuple] = []      # (arrival, order, receiver, message): delivered later
        self._order = itertools.count()

        self.time = 0.0

    def reset(self, seed: int | None = None) -> None:
        "Start over (e.g. a new episode): clock, queues, inboxes and metrics; `seed` reseeds the protocol and the backend."
        self.time = 0.0
        self._inbox.clear()
        self._timed.clear()
        self._in_flight.clear()
        self.protocol.reset(seed)
        self.backend.reset(seed)
        self.metrics.reset()

    def send(
        self,
        message: Message, # The message
        at: float | None = None, # When it is sent (None: with the next step); not before the network's time
    ) -> None:
        "Hand a message to the protocol, now or at time `at`."

        if message.sender not in self.nodes:
            raise ValueError(
                f"Unknown sender: {message.sender}"
            )

        if message.receiver not in self.nodes:
            raise ValueError(
                f"Unknown receiver: {message.receiver}"
            )

        if not self.topology.is_connected(
            message.sender,
            message.receiver,
        ):
            raise ValueError(
                f"No link from "
                f"{message.sender} to "
                f"{message.receiver}"
            )

        if at is None:
            self.protocol.submit(
                message,
                self.time,
            )
        elif at < self.time:
            raise ValueError(f"cannot send at {at}: the network is already at {self.time}")
        else:
            heapq.heappush(self._timed, (at, next(self._order), message))

    def step(
        self,
        dt: float = 1.0, # How far the clock moves
    ) -> List[TransmissionResult]:
        "Move the clock by `dt` and transmit (see `step_to`)."
        return self.step_to(self.time + dt)

    def step_to(
        self,
        time: float, # The new time (not before the network's time)
    ) -> List[TransmissionResult]:
        "Move the clock to `time`: messages sent for earlier times go out at their times, then the step's transmissions."
        if time < self.time:
            raise ValueError(f"cannot step back from {self.time} to {time}")
        results = []
        while self._timed and self._timed[0][0] < time:
            at = self._timed[0][0]
            while self._timed and self._timed[0][0] == at:
                self.protocol.submit(heapq.heappop(self._timed)[2], at)
            self.time = at
            results += self._transmit(at)
        while self._timed and self._timed[0][0] == time:
            self.protocol.submit(heapq.heappop(self._timed)[2], time)
        self.time = time
        results += self._transmit(time)
        return results

    def _transmit(self, time: float) -> List[TransmissionResult]:
        "One round: the protocol's transmissions at `time`, all at once."

        self.backend.step(time)

        transmissions = self.protocol.schedule(
            time
        )

        # the round's transmissions happen at the same time: the backend gets them all at once
        links = [
            self.topology.get_link(
                transmission.sender,
                transmission.receiver,
            )
            for transmission in transmissions
        ]

        results = self.backend.transmit_all(
            list(zip(transmissions, links))
        )

        for transmission, result in zip(transmissions, results):

            self.metrics.record(
                transmission,
                result,
            )

            if result.success:

                arrival = transmission.start_time + result.latency if self.delayed_delivery else time

                # what the receiver gets is what the channel delivered (it may differ from what was sent)
                message = replace(transmission.message, payload=result.payload,
                                  metadata={**transmission.message.metadata, 'received_at': arrival})

                if self.delayed_delivery:
                    heapq.heappush(self._in_flight, (arrival, next(self._order), transmission.receiver, message))
                else:
                    self._inbox[transmission.receiver].append(message)

        # acknowledgements: the protocol learns what got through (e.g. to retransmit)
        self.protocol.feedback(results)

        return results

    def receive(
        self,
        node_id: str, # The receiving node
    ) -> List[Message]:
        "The messages that have reached `node_id` by now, in the order they arrived."

        while self._in_flight and self._in_flight[0][0] <= self.time:
            _, _, receiver, message = heapq.heappop(self._in_flight)
            self._inbox[receiver].append(message)

        messages = self._inbox[node_id]

        self._inbox[node_id] = []

        return messages

    @property
    def in_flight(self) -> int:
        "Messages transmitted that have not arrived yet (with `delayed_delivery`)."
        return len(self._in_flight)

## Time

The network has a clock, `time`. `step(dt)` moves it by `dt`, and `step_to(time)` moves it to a given time. Each
move ends with a *round*: the protocol's transmissions at that time, simulated together by the backend.

* **Sending:** `send(message)` hands a message to the protocol for the next round. `send(message, at=t)` sends it at
  time `t`. When the clock passes `t`, there is a round at exactly `t`. This lets an application send at any time,
  e.g. a reply as soon as a request has arrived and been processed.
* **Receiving:** by default, a message reaches its receiver in the round it is transmitted. With
  `delayed_delivery=True`, it arrives at its transmission's start + latency (e.g. the airtime of its packets and
  their retries). `receive` then returns only what has arrived by the network's time, and `in_flight` counts the
  rest. Delivered messages carry their arrival time in `metadata['received_at']`.

With slotted protocols (e.g. `TDMAProtocol`), every round is a slot, including the rounds of messages sent `at` a
time.

In [ ]:
show_doc(Network.send)

In [ ]:
show_doc(Network.step_to)

In [ ]:
show_doc(Network.receive)

### Tests

In [ ]:
from comm_core.channel import AWGNChannel, Channel, IdentityChannel
from comm_core.backend import AnalyticalBackend
from comm_core.core import Message, TransmissionResult
from comm_core.protocol import IdealProtocol
from comm_core.topology import FullMeshTopology, StarTopology

In [ ]:
nodes = [Node(n) for n in ('a', 'b', 'c')]
net = Network(nodes, FullMeshTopology(['a', 'b', 'c']), IdealProtocol(), AnalyticalBackend(IdentityChannel()))
net.send(Message('a', 'b', payload=[1, 2], size_bits=64))
net.send(Message('c', 'b', payload=[3], size_bits=32))
results = net.step()
assert [r.success for r in results] == [True, True] and net.receive('b')[1].payload == [3]
assert net.receive('b') == [] and net.metrics.summary().bits_received == 96
# the receiver gets what the channel delivered
class Flip(Channel):
    def transmit(self, transmission, link):
        return TransmissionResult(success=True, transmission=transmission, payload=-transmission.message.payload)
net = Network(nodes, FullMeshTopology(['a', 'b', 'c']), IdealProtocol(), AnalyticalBackend(Flip()))
net.send(Message('a', 'c', payload=5))
net.step()
assert net.receive('c')[0].payload == -5
# reset: a new episode, replayed with the same seed
def episode(net, seed):
    net.reset(seed)
    out = []
    for _ in range(20):
        net.send(Message('a', 'b', payload=0))
        out += [r.success for r in net.step()]
    return out
net = Network(nodes, FullMeshTopology(['a', 'b', 'c']), IdealProtocol(), AnalyticalBackend(AWGNChannel(5, 0.5)))
first = episode(net, 0)
assert episode(net, 0) == first and net.time == 20.0 and net.metrics.summary().transmissions == 20
# messages need a link
net = Network(nodes, StarTopology('a', ['a', 'b', 'c']), IdealProtocol(), AnalyticalBackend(IdentityChannel()))
try:
    net.send(Message('b', 'c', payload=0)); raise AssertionError('no error')
except ValueError as e:
    assert 'No link' in str(e)

Delayed delivery and messages sent at given times:

In [ ]:
from comm_core.protocol import TDMAProtocol

In [ ]:
class Slow(Channel):
    "Everything arrives, 0.25 time units after it is sent."
    def transmit(self, transmission, link):
        return TransmissionResult(success=True, transmission=transmission, payload=transmission.message.payload, latency=0.25)

net = Network(nodes, FullMeshTopology(['a', 'b', 'c']), IdealProtocol(), AnalyticalBackend(Slow()), delayed_delivery=True)
net.send(Message('a', 'b', payload='hi'))
net.step(dt=0.1)                                   # transmitted at 0.1, arrives at 0.35
assert net.receive('b') == [] and net.in_flight == 1
net.step(dt=0.1)
assert net.receive('b') == []
net.step(dt=0.2)                                   # 0.4
m, = net.receive('b')
assert m.payload == 'hi' and abs(m.metadata['received_at'] - 0.35) < 1e-12 and net.in_flight == 0
# sent at given times: a round at each of them, in order
net.send(Message('b', 'c', payload=2), at=0.55)
net.send(Message('a', 'c', payload=1), at=0.45)
results = net.step_to(0.6)
assert [(r.transmission.start_time, r.transmission.message.payload) for r in results] == [(0.45, 1), (0.55, 2)]
assert [m.payload for m in net.receive('c')] == []            # arrive at 0.7 and 0.8
net.step_to(0.75)
assert [m.payload for m in net.receive('c')] == [1]
try:
    net.send(Message('a', 'b', payload=0), at=0.1); raise AssertionError('no error')
except ValueError:
    pass
try:
    net.step_to(0.5); raise AssertionError('no error')
except ValueError:
    pass
# a message sent for the very time of a step goes out in that step's round (one round, one slot)
tdma = TDMAProtocol(['a'])
net = Network(nodes, FullMeshTopology(['a', 'b', 'c']), tdma, AnalyticalBackend(IdentityChannel()))
net.send(Message('a', 'b', payload=1), at=1.0)
assert len(net.step(dt=1.0)) == 1 and tdma.slot == 1

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()